# Session 12: Visualization and Reporting

**Module 3: Programming for Biological Data**  
**Date:** February 11, 2026 | 18:30 – 21:30  
**Instructor:** Dr. Haogao Gu

---

## Learning Objectives

By the end of this session, you will be able to:
1. Use **ggtree** for advanced tree visualization
2. Add **metadata** to phylogenetic trees
3. Create **publication-quality** figures
4. Setup a **Local R Development Environment**

In [ ]:
# ============================================
# STANDARD SETUP PROTOCOL
# ============================================
options(repos = c(CRAN = "https://cloud.r-project.org"))

if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

# Install ggtree (may take a few minutes)
if (!requireNamespace("ggtree", quietly = TRUE))
  BiocManager::install("ggtree", update = FALSE, ask = FALSE)
if (!requireNamespace("ggplot2", quietly = TRUE))
  install.packages("ggplot2")
if (!requireNamespace("ape", quietly = TRUE))
  install.packages("ape")

library(ggtree)
library(ggplot2)
library(ape)

cat("✅ All packages loaded!")

---

# Part 1: Advanced Tree Visualization with ggtree

## 40 minutes

---

## 1.1 What is ggtree?

**ggtree** extends ggplot2 for phylogenetic trees:

- Beautiful, publication-ready visualizations
- Add metadata (colors, annotations)
- Multiple layout options
- Follows ggplot2 grammar

## 1.2 Creating a Demo Tree

In [ ]:
# Create a demo tree
tree_text <- "((Wuhan:0.01,Alpha:0.02):0.01,((Delta:0.03,Patient_SZ:0.01):0.02,(Omicron:0.04,(Patient_HK1:0.01,Patient_HK2:0.01):0.02):0.01):0.01);"
tree <- read.tree(text = tree_text)

# Basic ape plot
plot(tree, main = "Basic ape plot")

## 1.3 Basic ggtree

In [ ]:
# Basic ggtree
ggtree(tree) +
  geom_tiplab() +
  ggtitle("Basic ggtree")

## 1.4 Tree Layouts

In [ ]:
# Rectangular (default)
p1 <- ggtree(tree) + geom_tiplab() + ggtitle("Rectangular")

# Circular
p2 <- ggtree(tree, layout = "circular") + geom_tiplab() + ggtitle("Circular")

# Display
print(p1)

In [ ]:
# Circular layout
ggtree(tree, layout = "circular") +
  geom_tiplab(aes(angle = angle), hjust = -0.1) +
  ggtitle("Circular Layout")

## 1.5 Adding Metadata

In [ ]:
# Create metadata
metadata <- data.frame(
  label = c("Wuhan", "Alpha", "Delta", "Omicron", "Patient_SZ", "Patient_HK1", "Patient_HK2"),
  Location = c("China", "UK", "India", "South_Africa", "Shenzhen", "Hong_Kong", "Hong_Kong"),
  Type = c("Reference", "Reference", "Reference", "Reference", "Patient", "Patient", "Patient")
)

print(metadata)

In [ ]:
# Add metadata to tree
ggtree(tree) %<+% metadata +
  geom_tiplab(aes(color = Location)) +
  scale_color_brewer(palette = "Set1") +
  ggtitle("Tree Colored by Location")

In [ ]:
# Color by Type
ggtree(tree) %<+% metadata +
  geom_tiplab(aes(color = Type), fontface = "bold") +
  geom_tippoint(aes(color = Type), size = 3) +
  scale_color_manual(values = c("Reference" = "blue", "Patient" = "red")) +
  ggtitle("References vs Patients")

## 1.6 Adding Scale Bar

In [ ]:
# Add scale bar
ggtree(tree) %<+% metadata +
  geom_tiplab(aes(color = Location)) +
  geom_treescale(x = 0, y = 1) +
  scale_color_brewer(palette = "Set1") +
  theme_tree2()  # Adds axis

## 1.7 Highlighting Clades

In [ ]:
ggtree(tree)$data

In [ ]:
# Highlight a clade
ggtree(tree) %<+% metadata +
  geom_tiplab() +
  geom_hilight(node = 13, fill = "lightblue", alpha = 0.3) +  # HK cluster
  ggtitle("HK Cluster Highlighted")

## 1.8 Saving Figures

**ggsave()** is the standard way to export plots. It supports PNG, PDF, SVG, etc.

```r
ggsave("tree.png", width = 8, height = 6)
ggsave("tree.pdf", width = 8, height = 6)
```

In [ ]:
# Create final figure and save
p <- ggtree(tree) %<+% metadata +
  geom_tiplab(aes(color = Location)) +
  scale_color_brewer(palette = "Set1") +
  theme_tree2()

ggsave("final_tree.png", p, width = 8, height = 6)
cat("Saved final_tree.png")

---

# Part 2: Transitioning to Local R Development

## 40 minutes

So far, we have used Google Colab. To continue your bioinformatics journey, you need to set up R on your own computer.

---

## 2.1 The RStudio IDE

**RStudio** is the standard interface for R. It has 4 panes:
1. **Source:** Where you write scripts
2. **Console:** Where code runs (like Colab cells)
3. **Environment:** Your variables (dataframes, etc.)
4. **Files/Plots:** File browser and plot viewer

**Installation:**
1. Install **R** (CRAN)
2. Install **RStudio** (Posit)

## 2.2 R Projects (.Rproj)

**Never use `setwd()` again!**

An **R Project** sets the working directory to the project folder automatically. This makes your code portable (works on your Mac and your colleague's PC).

**To create:** `File -> New Project -> New Directory`

## 2.3 Managing Files Locally

In local development, file paths matter. Use **relative paths** (relative to your project folder).

**Standard Structure:**
- `my_project/`
  - `my_project.Rproj`
  - `data/` (Raw data)
  - `scripts/` (Analysis code)
  - `results/` (Plots and tables)

Let's simulate this structure here:

In [ ]:
# Create directories
dir.create("data", showWarnings = FALSE)
dir.create("scripts", showWarnings = FALSE)
dir.create("results", showWarnings = FALSE)

cat("Directories created!\n")
list.files()

## 2.4 Scripts vs Notebooks

- **Notebooks (.ipynb/Rmd):** Good for teaching, reports, and exploration.
- **Scripts (.R):** Good for reusable pipelines and automation.

Let's write a simple R script that generates a plot.

In [ ]:
script_content <- '
# Load library
library(ggplot2)

# Create data
df <- data.frame(x = 1:10, y = (1:10)^2)

# Plot
p <- ggplot(df, aes(x, y)) + geom_line() + ggtitle("Generated via Script")

# Save
ggsave("results/script_plot.png", p)
print("Plot saved to results/script_plot.png")
'

# Write this to a file
writeLines(script_content, "scripts/analysis.R")
cat("Script saved locally.")

## 2.5 Running Scripts

You can run an entire script using the `source()` command.

In [ ]:
# Run the script we just made
source("scripts/analysis.R")

# Check if result exists
file.exists("results/script_plot.png")

---

# Key Takeaways

1. **ggtree** creates beautiful, customizable phylogenetic trees.
2. **Local R Development** uses RStudio + R Projects.
3. **Organize your files!** Use `data`, `scripts`, `results` folders.
4. **Relative paths** make your project shareable.
5. **Scripts (.R)** automate analysis pipelines.

---

## Now proceed to the Final Tutorial! 🎓